In [ ]:
import fitz  # PyMuPDF
import os
import re
import numpy as np
from PIL import Image
import cv2
import shutil
import gc
import zipfile
from tqdm import tqdm

# ==========================================
# 1. PATH SETUP
# ==========================================
INPUT_DIRS = [r"D:\DEPI\Freelance\PDF Converter (real state)\input"]

BASE_OUTPUT = r"D:\DEPI\Freelance\PDF Converter (real state)\output"
BROCHURE_BASE = "./working_output/Brochures"

for root, dirs, files in os.walk("./input_pdfs"):
    for file in files:
        if file.lower().endswith(".pdf"):
            if root not in INPUT_DIRS:
                INPUT_DIRS.append(root)

print(f"Input directories: {INPUT_DIRS}")

# ==========================================
# SKIP LIST
# ==========================================
ALREADY_DONE = {
    "Marina 8", "Azha North Coast", "Masaya", "Marsa Baghush",
    "June", "Ogami Ras El Hekma", "Mazarine", "The Med",
    "Direction White", "Cali Coast Ras El Hekma", "Hacienda Bay",
    "Hacienda Heneish", "Ramla", "Hyde Park North", "Silversands",
    "Shamasi", "Fouka Bay", "LVLS Mountain View", "Safia Ras El Hekma",
    "Jefaira", "The Waterway North Coast", "Telal North Coast",
    "Caesar Sodic", "Naia Bay",
}
ALREADY_DONE_NORM = {n.lower().strip().replace(" ", "_") for n in ALREADY_DONE}
print(f"Skipping {len(ALREADY_DONE)} already-processed compounds.")


# ==========================================
# 2. NATIVE TEXT EXTRACTION
# ==========================================
def get_text_universal(page, clip_rect=None):
    """Blazing fast native text extraction via PyMuPDF."""
    if clip_rect:
        return page.get_text("text", clip=clip_rect)
    return page.get_text("text")

def get_text_with_positions(page):
    """Extracts words with normalized coordinates."""
    words = page.get_text("words")
    rect = page.rect
    w, h = rect.width, rect.height
    tokens = []
    for w_info in words:
        text = w_info[4].lower().strip()
        if not text: continue
        cx = (w_info[0] + w_info[2]) / 2 / w
        cy = (w_info[1] + w_info[3]) / 2 / h
        tokens.append((text, cx, cy, 1.0))
    return tokens

# ==========================================
# 3. IMAGE CONTENT CLASSIFIER
# ==========================================
def image_is_photo(pil_img, saturation_threshold=0.25, photo_pixel_ratio=0.30):
    thumb  = pil_img.resize((150, 150))
    img_np = np.array(thumb).astype(np.uint8)
    hsv    = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)
    sat    = hsv[:, :, 1] / 255.0
    return (sat > saturation_threshold).sum() / (150 * 150) > photo_pixel_ratio

def measure_line_density(pil_img):
    img_np = np.array(pil_img.resize((300, 300)))
    gray   = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY) if len(img_np.shape) == 3 else img_np
    edges  = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines  = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=40, minLineLength=20, maxLineGap=5)
    return (len(lines) if lines is not None else 0) / (300 * 300)

# ==========================================
# 4. OVERVIEW & SPLIT DETECTION
# ==========================================
OVERVIEW_UNIT_PATTERN = re.compile(r'(?:apartment|apt|unit|chalet|villa|townhouse|twin)\s*(\d{1,3})\s*[:\-]\s*(\d{2,4}(?:[.,]\d{1,2})?)\s*m', re.IGNORECASE)
OVERVIEW_UNIT_PATTERN_ALT = re.compile(r'(?:apartment|apt|unit|chalet|villa|townhouse|twin)\s*(\d{1,3})\s+(\d{2,4}(?:[.,]\d{1,2})?)\s*m', re.IGNORECASE)
OVERVIEW_MIN_UNITS = 3

def detect_building_overview(text):
    matches = OVERVIEW_UNIT_PATTERN.findall(text)
    if len(matches) < OVERVIEW_MIN_UNITS:
        matches = OVERVIEW_UNIT_PATTERN_ALT.findall(text)
    if len(matches) < OVERVIEW_MIN_UNITS:
        return False, []

    units = []
    for unit_num_str, area_str in matches:
        try:
            unit_num  = int(unit_num_str)
            area_val  = int(float(area_str.replace(',', '.')))
            if 40 <= area_val <= 1200: units.append((unit_num, area_val))
        except ValueError: continue
    return (len(units) >= OVERVIEW_MIN_UNITS), units

def detect_and_split_units(pil_img, tokens):
    img_width, img_height = pil_img.size
    anchor_pattern = re.compile(r'\b(type|ground|typical|first|second|third|fourth|fifth|penthouse|unit|bua|floor|plan)\b', re.IGNORECASE)

    anchor_x, anchor_y = [], []
    for (text, cx, cy, conf) in tokens:
        if anchor_pattern.search(text):
            anchor_x.append(cx * img_width)
            anchor_y.append(cy * img_height)

    if len(anchor_x) < 2: return [(pil_img, None)], False

    h_spread = (max(anchor_x) - min(anchor_x)) / img_width
    v_spread = (max(anchor_y) - min(anchor_y)) / img_height

    def find_split(positions):
        s = sorted(positions)
        best_gap, split = 0, s[len(s) // 2]
        for k in range(len(s) - 1):
            gap = s[k + 1] - s[k]
            if gap > best_gap:
                best_gap, split = gap, int((s[k] + s[k + 1]) / 2)
        return split

    if h_spread > 0.35:
        split_x = find_split(anchor_x)
        m = int(img_width * 0.01)
        box1 = (0, 0, split_x + m, img_height)
        box2 = (split_x - m, 0, img_width, img_height)
        return [(pil_img.crop(box1), box1), (pil_img.crop(box2), box2)], True

    if v_spread > 0.35:
        split_y = find_split(anchor_y)
        m = int(img_height * 0.01)
        box1 = (0, 0, img_width, split_y + m)
        box2 = (0, split_y - m, img_width, img_height)
        return [(pil_img.crop(box1), box1), (pil_img.crop(box2), box2)], True

    return [(pil_img, None)], False

# ==========================================
# 5. SCORING & CLASSIFICATION
# ==========================================
VISUAL_STRONG_SIGNALS = [r'\babout\b', r'\bamenities\b', r'\blifestyle\b', r'\bgallery\b', r'\bintroduc', r'\bwelcome\b', r'\bour\s+vision\b', r'\bfacilities\b', r'\bpool\b']
MASTER_PLAN_SIGNALS = [r'\bmaster\s*plan\b', r'\bsite\s*plan\b', r'\blayout\s*plan\b', r'\bgeneral\s*plan\b', r'\blegend\b', r'\bzoning\b']
FLOOR_PLAN_STRONG_SIGNALS = [r'\bunit\s*area\b', r'\bbua\b', r'\bfloor\s*plan\b', r'\bground\s*floor\b', r'\bbedroom\b', r'\bbathroom\b', r'\bkitchen\b', r'\bm2\b', r'\bsqm\b']

def score_page_category(text_cleaned, is_photo, line_density):
    scores = {"Floor_Plans": 0, "Master_Plans": 0, "Visuals": 0}
    if is_photo: scores["Visuals"] += 4
    if line_density > 0.003: scores["Floor_Plans"] += 3
    elif line_density < 0.001: scores["Visuals"] += 2

    for pattern in VISUAL_STRONG_SIGNALS:
        if re.search(pattern, text_cleaned): scores["Visuals"] += 3
    for pattern in MASTER_PLAN_SIGNALS:
        if re.search(pattern, text_cleaned): scores["Master_Plans"] += 3
    for pattern in FLOOR_PLAN_STRONG_SIGNALS:
        if re.search(pattern, text_cleaned): scores["Floor_Plans"] += 2

    max_score = max(scores.values())
    for cat in ["Floor_Plans", "Master_Plans", "Visuals"]:
        if scores[cat] == max_score: return cat

AREA_KEYWORDS = [(r'unit\s*area', 10), (r'bua', 10), (r'built[\s\-]*up', 10), (r'gross\s*area', 9), (r'total\s*area', 9)]

def extract_area_scored(text_cleaned):
    cleaned = re.sub(r'\d+[\.,]?\d*\s*[x×*]\s*\d+[\.,]?\d*', '', text_cleaned)
    candidates = {}

    def add_candidate(val_str, score):
        try:
            v = int(float(str(val_str).replace(',', '.')))
            if 40 <= v <= 1200 and not (2015 <= v <= 2035):
                candidates[v] = candidates.get(v, 0) + score
        except: pass

    for lo, hi in re.findall(r'(\d{2,4}(?:[.,]\d{1,2})?)\s*(?:m2|sqm|sq\.m|m²|m)?\s*(?:to|–|-)\s*(\d{2,4}(?:[.,]\d{1,2})?)\s*(?:m2|sqm|sq\.m|m²|m)', cleaned):
        add_candidate(lo, 12); add_candidate(hi, 8)
    for m in re.findall(r'starts?\s*from\s*(\d{2,4}(?:[.,]\d{1,2})?)', cleaned): add_candidate(m, 12)
    for pattern, score in AREA_KEYWORDS:
        for m in re.findall(rf'(?:{pattern})[^0-9]{{0,20}}(\d{{2,4}}(?:[.,]\d{{1,2}})?)', cleaned): add_candidate(m, score)
    for m in re.findall(r'(\d{2,4}(?:[.,]\d{1,2})?)\s*(?:m2|sqm|sq\.m|m²|\bm\b)', cleaned): add_candidate(m, 7)
    
    stripped = re.sub(r'p\d+\b', '', cleaned)
    stripped = re.sub(r'(?:terrace|roof|garden|plot|balcony|parking)\s*(?:area)?\s*\d+\s*(?:m|sqm|m2)?', '', stripped)
    for m in re.findall(r'(?<!\d)([3-9]\d|\d{3,4})(?!\d)', stripped): add_candidate(m, 1)

    if not candidates: return 0
    return sorted(candidates.items(), key=lambda kv: (-kv[1], abs(kv[0] - 150)))[0][0]

UNIT_KEYWORDS = [
    ("chalet", "Ground_Chalet", "Typical_Chalet"), ("penthouse", "Penthouse", "Penthouse"),
    ("standalone", "StandAlone_Villa", "StandAlone_Villa"), ("villa", "StandAlone_Villa", "StandAlone_Villa"),
    ("town house", None, None), ("townhouse", None, None), ("twin house", "TwinHouse", "TwinHouse"),
    ("hotel", "Ground_Hotel_Apartment", "Typical_Hotel_Apartment"), ("duplex", "Ground_Duplex", "Typical_Duplex"),
]

def ocr_classify_unit_type(text_cleaned):
    is_ground  = any(x in text_cleaned for x in ['ground', 'grd', 'g floor'])
    is_typical = any(x in text_cleaned for x in ['typical', 'typ.'])
    is_corner  = 'corner' in text_cleaned
    s = text_cleaned.replace(" ", "")

    for keyword, ground_type, default_type in UNIT_KEYWORDS:
        if keyword in text_cleaned or keyword.replace(" ", "") in s:
            if ground_type is None: return "TownHouse_Corner" if is_corner else "TownHouse_Middle"
            return ground_type if is_ground else default_type

    if is_ground: return "Ground_Apartment"
    if is_typical: return "Typical_Apartment"
    beds = re.findall(r'(\d)\s*-?\s*(?:bed|br)\b', text_cleaned)
    return f"{beds[0]}Bed_Apartment" if beds else "Typical_Apartment"

def analyze_page(text, pil_img=None):
    text_lower   = text.lower().replace("\n", " ").replace("_", " ").replace(":",  " ").replace("-", " ")
    text_cleaned = re.sub(r'\d+[\.,]?\d*\s*[x×*]\s*\d+[\.,]?\d*', '', text_lower)
    is_photo     = image_is_photo(pil_img) if pil_img else False
    line_density = measure_line_density(pil_img) if pil_img else 0.0
    category     = score_page_category(text_cleaned, is_photo, line_density)

    if category == "Master_Plans": return "Master_Plans", "Master_Plan"
    if category == "Visuals":
        if "location" in text_cleaned or "site" in text_cleaned: return "Visuals", "Location_Map"
        if any(re.search(p, text_cleaned) for p in [r'\babout\b', r'\bstory\b', r'\bvision\b']): return "Visuals", "About_Page"
        if any(re.search(p, text_cleaned) for p in [r'\bfacilit', r'\bgym\b', r'\bpool\b']): return "Visuals", "Amenities"
        return "Visuals", "Visual"

    ocr_area = extract_area_scored(text_cleaned)
    if ocr_area == 0:
        if any(x in text_cleaned for x in ["bedroom", "reception", "living", "kitchen"]):
            return "Floor_Plans", f"{ocr_classify_unit_type(text_cleaned)}_AreaMissing"
        return "Visuals", "Visual"

    unit_type = ocr_classify_unit_type(text_cleaned)
    label     = f"{unit_type}_{ocr_area}m"
    print(f"    Extracted type: {unit_type} {ocr_area}m")

    return "Floor_Plans", label

# ==========================================
# 6. TRACKER & ZIP HELPERS
# ==========================================
class UnitIndexTracker:
    def __init__(self):
        self._counter, self._label_to_index, self._label_last_page = 0, {}, {}

    def get(self, label: str, page_num: int, is_multi_page: bool) -> int:
        if label in self._label_to_index and is_multi_page and page_num <= self._label_last_page[label] + 2:
            self._label_last_page[label] = page_num
            return self._label_to_index[label]
        self._counter += 1
        self._label_to_index[label] = self._counter
        self._label_last_page[label] = page_num
        return self._counter

    def next(self) -> int:
        self._counter += 1
        return self._counter

def save_page_zip(project_dir, project_name, page_num):
    zip_path = os.path.join(BASE_OUTPUT, f"{project_name}_p{page_num}.zip")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for cat_folder in ["Floor_Plans", "Master_Plans", "Visuals"]:
            folder = os.path.join(project_dir, cat_folder)
            if not os.path.isdir(folder): continue
            for fname in os.listdir(folder):
                if f"_p{page_num}." in fname:
                    zf.write(os.path.join(folder, fname), arcname=os.path.join(cat_folder, fname))

# ==========================================
# 7. BROCHURE COPIER
# ==========================================
def copy_to_brochure_folder(pdf_path, filename):
    os.makedirs(BROCHURE_BASE, exist_ok=True)
    destination = os.path.join(BROCHURE_BASE, filename)
    shutil.copy2(pdf_path, destination)
    print(f"  Copied PDF to Brochures folder: {destination}")

# ==========================================
# 8. MAIN PROCESSING LOOP
# ==========================================
def process_all_files():
    if os.path.exists(BASE_OUTPUT): shutil.rmtree(BASE_OUTPUT)

    all_pdfs = [os.path.join(d, f) for d in INPUT_DIRS if os.path.isdir(d) for f in os.listdir(d) if f.lower().endswith('.pdf')]
    print(f"\nTotal PDFs found: {len(all_pdfs)}")

    for pdf_path in all_pdfs:
        pdf_filename = os.path.basename(pdf_path)
        project_name = os.path.splitext(pdf_filename)[0].replace(" ", "_")
        print(f"\n{'='*60}\nProcessing: {project_name}")

        if project_name.lower().strip() in ALREADY_DONE_NORM:
            print(f"  SKIPPED (already done)")
            continue
        
        # --- EXECUTE BROCHURE COPY ---
        copy_to_brochure_folder(pdf_path, pdf_filename)

        project_dir = os.path.join(BASE_OUTPUT, project_name)
        for cat in ["Floor_Plans", "Master_Plans", "Visuals"]:
            os.makedirs(os.path.join(project_dir, cat), exist_ok=True)

        tracker = UnitIndexTracker()
        doc = fitz.open(pdf_path)

        for i in tqdm(range(len(doc)), desc=project_name):
            try:
                page = doc.load_page(i)
                pix = page.get_pixmap(matrix=fitz.Matrix(3.0, 3.0))
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                page_num = i + 1

                # FAST TEXT EXTRACTION (Bypassing OCR)
                text = get_text_universal(page)
                tokens = get_text_with_positions(page)

                is_overview, overview_units = detect_building_overview(text)

                if is_overview:
                    text_cleaned = re.sub(r'\d+[\.,]?\d*\s*[x×*]\s*\d+[\.,]?\d*', '', text.lower().replace("\n"," ").replace("_"," ").replace(":"," ").replace("-"," "))
                    for unit_num, area_val in overview_units:
                        label = f"{ocr_classify_unit_type(text_cleaned)}_{area_val}m"
                        idx, safe_label = tracker.next(), label[:100]
                        filename = f"{safe_label}_{idx}_p{page_num}.png"
                        img.save(os.path.join(project_dir, "Floor_Plans", filename), optimize=True)

                else:
                    sub_images_data, is_multi = detect_and_split_units(img, tokens)
                    if is_multi:
                        for sub_img, pil_box in sub_images_data:
                            # Map PIL pixels back to PDF points for text extraction
                            pdf_box = fitz.Rect(pil_box[0]/3.0, pil_box[1]/3.0, pil_box[2]/3.0, pil_box[3]/3.0) if pil_box else None
                            sub_text = get_text_universal(page, clip_rect=pdf_box)
                            
                            cat, lbl = analyze_page(sub_text, pil_img=sub_img)
                            is_mp = any(t in lbl for t in ["TownHouse_Corner", "TownHouse_Middle", "TwinHouse", "StandAlone_Villa", "Ground_Duplex", "Typical_Duplex", "Ground_Chalet", "Typical_Chalet"])
                            idx, safe_label = tracker.get(lbl, page_num, is_mp), lbl[:100]
                            sub_img.save(os.path.join(project_dir, cat, f"{safe_label}_{idx}_p{page_num}.png"), optimize=True)
                    else:
                        cat, lbl = analyze_page(text, pil_img=img)
                        is_mp = any(t in lbl for t in ["TownHouse_Corner", "TownHouse_Middle", "TwinHouse", "StandAlone_Villa", "Ground_Duplex", "Typical_Duplex", "Ground_Chalet", "Typical_Chalet"])
                        idx, safe_label = tracker.get(lbl, page_num, is_mp), lbl[:100]
                        img.save(os.path.join(project_dir, cat, f"{safe_label}_{idx}_p{page_num}.png"), optimize=True)

                save_page_zip(project_dir, project_name, page_num)
                del img, pix
                gc.collect()

            except Exception as e:
                print(f"  Error on page {i + 1}: {e}")

        doc.close()
        shutil.make_archive(os.path.join(BASE_OUTPUT, f"{project_name}_FULL"), 'zip', project_dir)

    shutil.make_archive(os.path.join(os.path.dirname(BASE_OUTPUT), "All_Projects_Output"), 'zip', BASE_OUTPUT)
    print("\nAll done!")

if INPUT_DIRS:
    process_all_files()

In [ ]:
import fitz  # PyMuPDF
import os
import re
import numpy as np
from PIL import Image
import cv2
import shutil
import gc
from tqdm import tqdm

# ==========================================
# 1. PATH SETUP
# ==========================================
INPUT_DIRS = [
    r"D:\DEPI\Freelance\PDF Converter (real state)\input"
]

BASE_OUTPUT = r"D:\DEPI\Freelance\PDF Converter (real state)\output"
BROCHURE_BASE = "./working_output/Brochures"

print(f"Input directories: {INPUT_DIRS}")

# ==========================================
# SKIP LIST
# ==========================================
ALREADY_DONE = {
    "Marina 8", "Azha North Coast", "Masaya", "Marsa Baghush",
    "June", "Ogami Ras El Hekma", "Mazarine", "The Med",
    "Direction White", "Cali Coast Ras El Hekma", "Hacienda Bay",
    "Hacienda Heneish", "Ramla", "Hyde Park North", "Silversands",
    "Shamasi", "Fouka Bay", "LVLS Mountain View", "Safia Ras El Hekma",
    "Jefaira", "The Waterway North Coast", "Telal North Coast",
    "Caesar Sodic", "Naia Bay",
}

ALREADY_DONE_NORM = {
    n.lower().strip().replace(" ", "_")
    for n in ALREADY_DONE
}

print(f"Skipping {len(ALREADY_DONE)} already-processed compounds.")

# ==========================================
# 2. TEXT EXTRACTION
# ==========================================
def get_text_universal(page, clip_rect=None):
    if clip_rect:
        return page.get_text("text", clip=clip_rect)
    return page.get_text("text")


def get_text_with_positions(page):
    words = page.get_text("words")

    rect = page.rect
    w, h = rect.width, rect.height

    tokens = []

    for w_info in words:
        text = w_info[4].lower().strip()

        if not text:
            continue

        cx = (w_info[0] + w_info[2]) / 2 / w
        cy = (w_info[1] + w_info[3]) / 2 / h

        tokens.append((text, cx, cy, 1.0))

    return tokens


# ==========================================
# 3. IMAGE ANALYSIS
# ==========================================
def image_is_photo(pil_img, saturation_threshold=0.25, photo_pixel_ratio=0.30):

    thumb = pil_img.resize((150, 150))

    img_np = np.array(thumb).astype(np.uint8)

    hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)

    sat = hsv[:, :, 1] / 255.0

    return (
        (sat > saturation_threshold).sum() / (150 * 150)
        > photo_pixel_ratio
    )


def measure_line_density(pil_img):

    img_np = np.array(pil_img.resize((300, 300)))

    gray = (
        cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        if len(img_np.shape) == 3
        else img_np
    )

    edges = cv2.Canny(gray, 50, 150, apertureSize=3)

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=40,
        minLineLength=20,
        maxLineGap=5
    )

    return (
        len(lines) if lines is not None else 0
    ) / (300 * 300)


# ==========================================
# 4. OVERVIEW DETECTION
# ==========================================
OVERVIEW_UNIT_PATTERN = re.compile(
    r'(?:apartment|apt|unit|chalet|villa|townhouse|twin)\s*(\d{1,3})\s*[:\-]\s*(\d{2,4}(?:[.,]\d{1,2})?)\s*m',
    re.IGNORECASE
)

OVERVIEW_UNIT_PATTERN_ALT = re.compile(
    r'(?:apartment|apt|unit|chalet|villa|townhouse|twin)\s*(\d{1,3})\s+(\d{2,4}(?:[.,]\d{1,2})?)\s*m',
    re.IGNORECASE
)

OVERVIEW_MIN_UNITS = 3


def detect_building_overview(text):

    matches = OVERVIEW_UNIT_PATTERN.findall(text)

    if len(matches) < OVERVIEW_MIN_UNITS:
        matches = OVERVIEW_UNIT_PATTERN_ALT.findall(text)

    if len(matches) < OVERVIEW_MIN_UNITS:
        return False, []

    units = []

    for unit_num_str, area_str in matches:

        try:
            unit_num = int(unit_num_str)

            area_val = int(
                float(area_str.replace(',', '.'))
            )

            if 40 <= area_val <= 1200:
                units.append((unit_num, area_val))

        except ValueError:
            continue

    return (len(units) >= OVERVIEW_MIN_UNITS), units


# ==========================================
# 5. SMART SPLIT DETECTION
# ==========================================
def detect_and_split_units(pil_img, tokens):

    img_width, img_height = pil_img.size

    anchor_pattern = re.compile(
        r'\b(type|ground|typical|first|second|third|fourth|fifth|penthouse|unit|bua|floor|plan)\b',
        re.IGNORECASE
    )

    anchor_x = []
    anchor_y = []

    for (text, cx, cy, conf) in tokens:

        if anchor_pattern.search(text):
            anchor_x.append(cx * img_width)
            anchor_y.append(cy * img_height)

    if len(anchor_x) < 4:
        return [(pil_img, None)], False

    h_spread = (
        max(anchor_x) - min(anchor_x)
    ) / img_width

    v_spread = (
        max(anchor_y) - min(anchor_y)
    ) / img_height

    def find_split(positions):

        s = sorted(positions)

        best_gap = 0
        split = s[len(s)//2]

        for k in range(len(s)-1):

            gap = s[k+1] - s[k]

            if gap > best_gap:
                best_gap = gap
                split = int((s[k] + s[k+1]) / 2)

        return split

    def has_meaningful_content(crop_img):

        arr = np.array(crop_img)

        gray = (
            cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            if len(arr.shape) == 3
            else arr
        )

        non_white = np.sum(gray < 245)

        ratio = non_white / gray.size

        return ratio > 0.08

    # ======================================
    # HORIZONTAL SPLIT
    # ======================================
    if h_spread > 0.45:

        split_x = find_split(anchor_x)

        if (
            split_x < img_width * 0.25
            or split_x > img_width * 0.75
        ):
            return [(pil_img, None)], False

        margin = int(img_width * 0.01)

        box1 = (
            0,
            0,
            split_x + margin,
            img_height
        )

        box2 = (
            split_x - margin,
            0,
            img_width,
            img_height
        )

        crop1 = pil_img.crop(box1)
        crop2 = pil_img.crop(box2)

        if (
            has_meaningful_content(crop1)
            and has_meaningful_content(crop2)
        ):

            return [
                (crop1, box1),
                (crop2, box2)
            ], True

    # ======================================
    # VERTICAL SPLIT
    # ======================================
    if v_spread > 0.45:

        split_y = find_split(anchor_y)

        if (
            split_y < img_height * 0.25
            or split_y > img_height * 0.75
        ):
            return [(pil_img, None)], False

        margin = int(img_height * 0.01)

        box1 = (
            0,
            0,
            img_width,
            split_y + margin
        )

        box2 = (
            0,
            split_y - margin,
            img_width,
            img_height
        )

        crop1 = pil_img.crop(box1)
        crop2 = pil_img.crop(box2)

        if (
            has_meaningful_content(crop1)
            and has_meaningful_content(crop2)
        ):

            return [
                (crop1, box1),
                (crop2, box2)
            ], True

    return [(pil_img, None)], False


# ==========================================
# 6. CLASSIFICATION
# ==========================================
VISUAL_STRONG_SIGNALS = [
    r'\babout\b',
    r'\bamenities\b',
    r'\blifestyle\b',
    r'\bgallery\b',
    r'\bintroduc',
    r'\bwelcome\b',
    r'\bour\s+vision\b',
    r'\bfacilities\b',
    r'\bpool\b'
]

MASTER_PLAN_SIGNALS = [
    r'\bmaster\s*plan\b',
    r'\bsite\s*plan\b',
    r'\blayout\s*plan\b',
    r'\bgeneral\s*plan\b',
    r'\blegend\b',
    r'\bzoning\b'
]

FLOOR_PLAN_STRONG_SIGNALS = [
    r'\bunit\s*area\b',
    r'\bbua\b',
    r'\bfloor\s*plan\b',
    r'\bground\s*floor\b',
    r'\bbedroom\b',
    r'\bbathroom\b',
    r'\bkitchen\b',
    r'\bm2\b',
    r'\bsqm\b'
]


def score_page_category(text_cleaned, is_photo, line_density):

    scores = {
        "Floor_Plans": 0,
        "Master_Plans": 0,
        "Visuals": 0
    }

    if is_photo:
        scores["Visuals"] += 4

    if line_density > 0.003:
        scores["Floor_Plans"] += 3

    elif line_density < 0.001:
        scores["Visuals"] += 2

    for pattern in VISUAL_STRONG_SIGNALS:
        if re.search(pattern, text_cleaned):
            scores["Visuals"] += 3

    for pattern in MASTER_PLAN_SIGNALS:
        if re.search(pattern, text_cleaned):
            scores["Master_Plans"] += 3

    for pattern in FLOOR_PLAN_STRONG_SIGNALS:
        if re.search(pattern, text_cleaned):
            scores["Floor_Plans"] += 2

    max_score = max(scores.values())

    for cat in scores:
        if scores[cat] == max_score:
            return cat


AREA_KEYWORDS = [
    (r'unit\s*area', 10),
    (r'bua', 10),
    (r'built[\s\-]*up', 10),
    (r'gross\s*area', 9),
    (r'total\s*area', 9)
]


def extract_area_scored(text_cleaned):

    cleaned = re.sub(
        r'\d+[\.,]?\d*\s*[x×*]\s*\d+[\.,]?\d*',
        '',
        text_cleaned
    )

    candidates = {}

    def add_candidate(val_str, score):

        try:
            v = int(float(str(val_str).replace(',', '.')))

            if 40 <= v <= 1200 and not (2015 <= v <= 2035):
                candidates[v] = candidates.get(v, 0) + score

        except:
            pass

    for m in re.findall(
        r'(\d{2,4}(?:[.,]\d{1,2})?)\s*(?:m2|sqm|sq\.m|m²|\bm\b)',
        cleaned
    ):
        add_candidate(m, 7)

    if not candidates:
        return 0

    return sorted(
        candidates.items(),
        key=lambda kv: (-kv[1], abs(kv[0] - 150))
    )[0][0]


UNIT_KEYWORDS = [
    ("chalet", "Ground_Chalet", "Typical_Chalet"),
    ("penthouse", "Penthouse", "Penthouse"),
    ("standalone", "StandAlone_Villa", "StandAlone_Villa"),
    ("villa", "StandAlone_Villa", "StandAlone_Villa"),
    ("town house", None, None),
    ("townhouse", None, None),
    ("twin house", "TwinHouse", "TwinHouse"),
    ("hotel", "Ground_Hotel_Apartment", "Typical_Hotel_Apartment"),
    ("duplex", "Ground_Duplex", "Typical_Duplex"),
]


def ocr_classify_unit_type(text_cleaned):

    is_ground = any(
        x in text_cleaned
        for x in ['ground', 'grd', 'g floor']
    )

    is_corner = 'corner' in text_cleaned

    s = text_cleaned.replace(" ", "")

    for keyword, ground_type, default_type in UNIT_KEYWORDS:

        if keyword in text_cleaned or keyword.replace(" ", "") in s:

            if ground_type is None:
                return (
                    "TownHouse_Corner"
                    if is_corner
                    else "TownHouse_Middle"
                )

            return (
                ground_type
                if is_ground
                else default_type
            )

    if is_ground:
        return "Ground_Apartment"

    return "Typical_Apartment"


def analyze_page(text, pil_img=None):

    text_lower = (
        text.lower()
        .replace("\n", " ")
        .replace("_", " ")
        .replace(":", " ")
        .replace("-", " ")
    )

    text_cleaned = re.sub(
        r'\d+[\.,]?\d*\s*[x×*]\s*\d+[\.,]?\d*',
        '',
        text_lower
    )

    is_photo = image_is_photo(pil_img) if pil_img else False

    line_density = (
        measure_line_density(pil_img)
        if pil_img
        else 0.0
    )

    category = score_page_category(
        text_cleaned,
        is_photo,
        line_density
    )

    if category == "Master_Plans":
        return "Master_Plans", "Master_Plan"

    if category == "Visuals":
        return "Visuals", "Visual"

    ocr_area = extract_area_scored(text_cleaned)

    if ocr_area == 0:
        return "Floor_Plans", "AreaMissing"

    unit_type = ocr_classify_unit_type(text_cleaned)

    label = f"{unit_type}_{ocr_area}m"

    print(f"    Extracted: {label}")

    return "Floor_Plans", label


# ==========================================
# 7. TRACKER
# ==========================================
class UnitIndexTracker:

    def __init__(self):

        self._counter = 0

        self._label_to_index = {}

        self._label_last_page = {}

    def get(self, label, page_num, is_multi_page):

        if (
            label in self._label_to_index
            and is_multi_page
            and page_num <= self._label_last_page[label] + 2
        ):

            self._label_last_page[label] = page_num

            return self._label_to_index[label]

        self._counter += 1

        self._label_to_index[label] = self._counter

        self._label_last_page[label] = page_num

        return self._counter

    def next(self):

        self._counter += 1

        return self._counter


# ==========================================
# 8. BROCHURE COPY
# ==========================================
def copy_to_brochure_folder(pdf_path, filename):

    os.makedirs(BROCHURE_BASE, exist_ok=True)

    destination = os.path.join(
        BROCHURE_BASE,
        filename
    )

    shutil.copy2(pdf_path, destination)

    print(f"  Copied brochure: {destination}")


# ==========================================
# 9. MAIN PROCESSING
# ==========================================
def process_all_files():

    if os.path.exists(BASE_OUTPUT):
        shutil.rmtree(BASE_OUTPUT)

    all_pdfs = []

    for d in INPUT_DIRS:

        if not os.path.isdir(d):
            continue

        for f in os.listdir(d):

            if f.lower().endswith(".pdf"):

                all_pdfs.append(
                    os.path.join(d, f)
                )

    print(f"\nTotal PDFs found: {len(all_pdfs)}")

    for pdf_path in all_pdfs:

        pdf_filename = os.path.basename(pdf_path)

        project_name = (
            os.path.splitext(pdf_filename)[0]
            .replace(" ", "_")
        )

        print(f"\n{'='*60}")
        print(f"Processing: {project_name}")

        if (
            project_name.lower().strip()
            in ALREADY_DONE_NORM
        ):
            print("  SKIPPED")
            continue

        copy_to_brochure_folder(
            pdf_path,
            pdf_filename
        )

        project_dir = os.path.join(
            BASE_OUTPUT,
            project_name
        )

        for cat in [
            "Floor_Plans",
            "Master_Plans",
            "Visuals"
        ]:

            os.makedirs(
                os.path.join(project_dir, cat),
                exist_ok=True
            )

        tracker = UnitIndexTracker()

        doc = fitz.open(pdf_path)

        for i in tqdm(
            range(len(doc)),
            desc=project_name
        ):

            try:

                page = doc.load_page(i)

                pix = page.get_pixmap(
                    matrix=fitz.Matrix(3.0, 3.0)
                )

                img = Image.frombytes(
                    "RGB",
                    [pix.width, pix.height],
                    pix.samples
                )

                page_num = i + 1

                text = get_text_universal(page)

                tokens = get_text_with_positions(page)

                is_overview, overview_units = detect_building_overview(text)

                # ==================================
                # OVERVIEW PAGE
                # ==================================
                if is_overview:

                    text_cleaned = re.sub(
                        r'\d+[\.,]?\d*\s*[x×*]\s*\d+[\.,]?\d*',
                        '',
                        text.lower()
                    )

                    for unit_num, area_val in overview_units:

                        label = (
                            f"{ocr_classify_unit_type(text_cleaned)}"
                            f"_{area_val}m"
                        )

                        idx = tracker.next()

                        safe_label = label[:100]

                        filename = (
                            f"{safe_label}_{idx}_p{page_num}.png"
                        )

                        img.save(
                            os.path.join(
                                project_dir,
                                "Floor_Plans",
                                filename
                            ),
                            optimize=True
                        )

                # ==================================
                # NORMAL PAGE
                # ==================================
                else:

                    sub_images_data, is_multi = detect_and_split_units(
                        img,
                        tokens
                    )

                    if is_multi:

                        processed_labels = set()

                        for sub_img, pil_box in sub_images_data:

                            pdf_box = fitz.Rect(
                                pil_box[0] / 3.0,
                                pil_box[1] / 3.0,
                                pil_box[2] / 3.0,
                                pil_box[3] / 3.0
                            )

                            sub_text = get_text_universal(
                                page,
                                clip_rect=pdf_box
                            )

                            cat, lbl = analyze_page(
                                sub_text,
                                pil_img=sub_img
                            )

                            if lbl in processed_labels:
                                continue

                            processed_labels.add(lbl)

                            is_mp = any(
                                t in lbl
                                for t in [
                                    "TownHouse_Corner",
                                    "TownHouse_Middle",
                                    "TwinHouse",
                                    "StandAlone_Villa",
                                    "Ground_Duplex",
                                    "Typical_Duplex",
                                    "Ground_Chalet",
                                    "Typical_Chalet"
                                ]
                            )

                            idx = tracker.get(
                                lbl,
                                page_num,
                                is_mp
                            )

                            safe_label = lbl[:100]

                            # SAVE ORIGINAL FULL IMAGE
                            img.save(
                                os.path.join(
                                    project_dir,
                                    cat,
                                    f"{safe_label}_{idx}_p{page_num}.png"
                                ),
                                optimize=True
                            )

                    else:

                        cat, lbl = analyze_page(
                            text,
                            pil_img=img
                        )

                        is_mp = any(
                            t in lbl
                            for t in [
                                "TownHouse_Corner",
                                "TownHouse_Middle",
                                "TwinHouse",
                                "StandAlone_Villa",
                                "Ground_Duplex",
                                "Typical_Duplex",
                                "Ground_Chalet",
                                "Typical_Chalet"
                            ]
                        )

                        idx = tracker.get(
                            lbl,
                            page_num,
                            is_mp
                        )

                        safe_label = lbl[:100]

                        img.save(
                            os.path.join(
                                project_dir,
                                cat,
                                f"{safe_label}_{idx}_p{page_num}.png"
                            ),
                            optimize=True
                        )

                del img
                del pix

                gc.collect()

            except Exception as e:

                print(f"  Error on page {i+1}: {e}")

        doc.close()

        # ONLY FULL PROJECT ZIP
        shutil.make_archive(
            os.path.join(
                BASE_OUTPUT,
                f"{project_name}_FULL"
            ),
            'zip',
            project_dir
        )

    print("\nAll done!")


# ==========================================
# RUN
# ==========================================
if INPUT_DIRS:
    process_all_files()